In [88]:
import numpy as np
import pandas as pd
import time

class display(object):
    """Display HTML representation of multiple objects"""
    template = """<div style="float: left; padding: 10px;">
    <p style='font-family:"Courier New", Courier, monospace'>{0}</p>{1}
    </div>"""
    def __init__(self, *args):
        self.args = args
        
    def _repr_html_(self):
        rendered = []
        for a in self.args:
            try:
                rendered.append(self.template.format(a, eval(a)._repr_html_()))
            except Exception as e:
                rendered.append(self.template.format(a, f"<pre>Error: {e}</pre>"))
        return '\n'.join(rendered)
    
    def __repr__(self):
        lines = []
        for a in self.args:
            try:
                lines.append(a + '\n' + repr(eval(a)))
            except Exception as e:
                lines.append(f"{a}\nError: {e}")
        return '\n\n'.join(lines)

# pandas.eval for efficient operations

In [89]:
df1 = pd.DataFrame(np.random.randn(10_000_000, 20))
df2 = pd.DataFrame(np.random.randn(10_000_000, 20))
start = time.time()
pd.eval('df1 / 2.3 + 5 * df2')
end = time.time()
print(f"Time taken for pd.eval: {(end - start) * 1000} milliseconds")
start = time.time()
df1 / 2.3 + 5 * df2
end = time.time()
print(f"Time taken for standard addition: {(end - start) * 1000} milliseconds")

Time taken for pd.eval: 2664.0729904174805 milliseconds
Time taken for standard addition: 2446.669340133667 milliseconds


## operators supported by pandas.eval
- Arithmetic operators: `+`, `-`, `*`, `/`, `**`, `%`, `//`
- Comparison operators: `==`, `!=`, `<`, `>`, `<=`, `>=`
- Bitwise operators: `&`, `|`
- object attributes: `df.column_name` or `df['column_name']`

In [96]:
df1, df2, df3, df4 = [pd.DataFrame(np.random.randn(5, 5), columns=list('ABCDE')) for _ in range(4)]

print(pd.eval('df1 + df2 - df3 * df4'))
print(pd.eval('(df1 >= df3) | (df2 <= df4)'))
print(pd.eval('df1.T'))

          A         B         C         D         E
0  1.602230 -0.137420 -1.749676  2.952505  0.000316
1  1.028347  2.252835  1.678717  0.551398 -2.132665
2 -2.694657  1.002336 -1.085781 -0.764903 -1.421775
3 -0.061578  0.852829  1.819955  1.162427 -0.013222
4 -1.265894 -2.004707  2.555809 -0.855211 -0.700889
       A      B      C      D      E
0   True   True   True   True   True
1   True  False  False  False   True
2  False   True   True   True   True
3   True   True   True  False   True
4   True   True   True   True  False
          0         1         2         3         4
A  2.374760  1.976652 -2.753782 -1.524160  0.933004
B -0.132064  0.319564  0.238995 -0.732152 -1.399387
C  0.095590  0.703431  0.404709  0.141370  1.021424
D  2.745169 -0.429716  0.155692  0.277123  0.761910
E  1.687452  0.838440 -1.205864  1.262879 -1.593592


# dataframe.eval for column-wise operations

In [105]:
df = df1.copy()
display('df', "df.eval('A + B * C').to_frame()")

df
          A         B         C         D         E
0  2.374760 -0.132064  0.095590  2.745169  1.687452
1  1.976652  0.319564  0.703431 -0.429716  0.838440
2 -2.753782  0.238995  0.404709  0.155692 -1.205864
3 -1.524160 -0.732152  0.141370  0.277123  1.262879
4  0.933004 -1.399387  1.021424  0.761910 -1.593592

df.eval('A + B * C').to_frame()
          0
0  2.362136
1  2.201444
2 -2.657059
3 -1.627665
4 -0.496363

## assignment in dataframe.eval

In [111]:
df = pd.DataFrame(np.random.randn(5, 3), columns=list('ABE'))
display('df', "df.eval('F = 99 * A + B * E')")

df
          A         B         E
0 -1.474520 -1.476869 -0.220566
1  0.023365 -0.368574  0.307860
2  0.111582  1.513710 -0.410581
3  1.297739  0.682806 -0.550468
4  0.865019  0.083829 -0.792810

df.eval('F = 99 * A + B * E')
          A         B         E           F
0 -1.474520 -1.476869 -0.220566 -145.651769
1  0.023365 -0.368574  0.307860    2.199668
2  0.111582  1.513710 -0.410581   10.425103
3  1.297739  0.682806 -0.550468  128.100291
4  0.865019  0.083829 -0.792810   85.570408

## local variables in dataframe.eval

In [112]:
num = 0x99999
display('df', "df.eval('F = A + @num')")

df
          A         B         E
0 -1.474520 -1.476869 -0.220566
1  0.023365 -0.368574  0.307860
2  0.111582  1.513710 -0.410581
3  1.297739  0.682806 -0.550468
4  0.865019  0.083829 -0.792810

df.eval('F = A + @num')
          A         B         E              F
0 -1.474520 -1.476869 -0.220566  629143.525480
1  0.023365 -0.368574  0.307860  629145.023365
2  0.111582  1.513710 -0.410581  629145.111582
3  1.297739  0.682806 -0.550468  629146.297739
4  0.865019  0.083829 -0.792810  629145.865019

# dataframe.query and instance query function

In [114]:
display('df', 'df.query("A > 0 and B < 0")', 'pd.eval("df.query(\'A > 0 and B < 0\')")')

,A,B,E
0,-1.474520,-1.476869,-0.220566
1,0.023365,-0.368574,0.307860
2,0.111582,1.513710,-0.410581
3,1.297739,0.682806,-0.550468
4,0.865019,0.083829,-0.792810
,A,B,E
1,0.023365,-0.368574,0.30786
,A,B,E
1,0.023365,-0.368574,0.30786


# performance: when to use these functions

if the data is too large, use the eval and query. the overhead in time and storage of new dataframes for traditional operations can add up. The eval and query functions are optimized for performance and can handle large datasets more efficiently. for small datasets, the performance is better in traditional operations.